# Parallax Risk — Monte Carlo moments and reproducibility

Phase 4. All inputs are synthetic. Financial calculations, streams, estimators and experiments are production modules; cells only organize the results. Install the locked development environment and use its Python kernel. Times are year fractions, rates are decimal/year, and spot is USD per underlying unit.

In [1]:
import sys

import pandas as pd

from parallax_risk import __version__
from parallax_risk.application.simulation_examples import moment_experiment, synthetic_inputs
from parallax_risk.common.canonical import canonical_value
from parallax_risk.domain.simulation.engine import MonteCarloEngine

print({"project": "Parallax Risk", "release": __version__, "interpreter": sys.executable})

{'project': 'Parallax Risk', 'release': '0.4.0', 'interpreter': 'C:\\Users\\vs250\\Documents\\ChatGPT\\Parallax Risk\\.venv\\Scripts\\python.exe'}


## Distribution checks
Exact marginal GBM and Vasicek transitions have known terminal moments. The sample variance below is descriptive; the standard error and approximate Student t confidence interval use independent paths.

In [2]:
experiment = moment_experiment()
pd.DataFrame(
    [
        {
            "model": "GBM",
            "mean": experiment.gbm.raw_mean,
            "reference_mean": experiment.gbm_reference_mean,
            "sample_variance": experiment.gbm.raw_descriptive_variance,
            "reference_variance": experiment.gbm_reference_variance,
            "standard_error": experiment.gbm.estimate.standard_error,
            "confidence_interval": experiment.gbm.estimate.confidence_interval,
        },
        {
            "model": "Vasicek",
            "mean": experiment.vasicek.raw_mean,
            "reference_mean": experiment.vasicek_reference_mean,
            "sample_variance": experiment.vasicek.raw_descriptive_variance,
            "reference_variance": experiment.vasicek_reference_variance,
            "standard_error": experiment.vasicek.estimate.standard_error,
            "confidence_interval": experiment.vasicek.estimate.confidence_interval,
        },
    ]
)

,model,mean,reference_mean,sample_variance,reference_variance,standard_error,confidence_interval
0,GBM,105.177111,105.127110,453.955067,451.028808,0.083227,"(105.01398485917485, 105.34023640360587)"
1,Vasicek,0.036627,0.036594,0.000276,0.000275,0.000065,"(0.036499524432350654, 0.03675398680688106)"


## Replay evidence
The root seed, stream/substream address, generator, normal transform, model, grid, draw order and library versions identify the computation. Run identity and timestamps do not drive randomness. Batch-size invariance is tested for the pinned environments. Broad bitwise compatibility across different numerical libraries or CPUs is not promised.

In [3]:
request, context = synthetic_inputs(paths=256, batch=64)
engine = MonteCarloEngine()
first = tuple(batch.values.hash for batch in engine.iter_batches(request))
second = tuple(batch.values.hash for batch in engine.iter_batches(request))
assert first == second
canonical_value(engine.metadata(request))

{'__type__': 'parallax_risk.domain.simulation.contracts.SimulationMetadata',
 'request_hash': 'b2cf7e46160be6b2fdf321e9b62aa4d4ecbbce841ad9a187d16e1d37042f639c',
 'engine_version': '0.4.0',
 'sequence': {'__type__': 'parallax_risk.domain.simulation.random.SequenceSpec',
  'key': {'__type__': 'parallax_risk.domain.simulation.random.StreamKey',
   'seed': 20251001,
   'stream': 0,
   'substream': 0},
  'kind': <SequenceKind.PSEUDO: 'pseudo'>,
  'antithetic': False,
  'sobol_bits': 30},
 'algorithm': 'PCG64DXSM.standard_normal',
 'normal_transform': 'numpy_ziggurat_float64',
 'numpy_version': '2.5.3',
 'scipy_version': '1.18.1',
 'python_version': '3.13.2',
 'platform': 'Windows-11-10.0.26200-SP0',
 'source_revision': None,
 'state_names': ['asset.s0'],
 'factor_names': ['asset.d0'],
 'state_units': ['USD/underlying_unit'],
 'measures': ['Q'],
 'times': [0.0, 1.0],
 'path_count': 256,
 'batch_size': 64,
 'correlation_hash': None,
 'layout': 'path,time,state; normals=path,step,pre-loading-

## Limits
Exact Gaussian rate transitions do not simulate their joint integrated discount factors. Heston uses an explicitly selected projected Euler scheme, whose projections and bias are reported. Finite-time discretization error, random sampling error, and model error are distinct. No exposure, netting, collateral or XVA workflow is implemented in Phase 4.

See [methodology](../docs/methodology/MONTE_CARLO.md) and [phase evidence](../docs/validation/phase-4.md).